Eventually, you'll want to take advantage of the automatization that SCOPE enables. In this tutorial, we will prepare a more advanced workflow than in Tutorial 4.  
Make sure you read and understood Tutorial 4 before starting with this one

# <div style="text-align: center"> Introduction </div>

---
Along these tutorials, we will see how <span style="color:blue">**SCOPE**</span> interacts with the different parts of the code interact to handle the execution of computational workflows. 

These are the topics covered in each tutorial:
1) The **System**, **Specie**, **Cell** and **Atom** classes  
2) The Computational workflow: **Branch**, **Workflow**, **Job**, and **Computation** classes  
3) The **State** class  
4) The **Data**, **Collection** and **VNM** classes
5) The **Input_data** class, and **scope input files**
6) How to Run <span style="color:blue">**SCOPE**</span> - Part 1: What to expect
7) How to Run <span style="color:blue">**SCOPE**</span> - Part 2: Simple Setup
8) How to Run <span style="color:blue">**SCOPE**</span> - Part 3: Detailed Actions
---

# <div style="text-align: center"> Tutorial 7: How to Run SCOPE, part 2</div>

Eventually, you'll want to run a computational workflow for a given system. To do so, you need: 

0) To **configure scope** in a computation cluster (i.e. a cluster with a job manager, SLURM or SGE). You can also configure scope in your local computer, to analyze the data, but you won't be able to submit computations. 
 
    To do it, you can run the script "**scope_config**" in either/both the local computer and the computation cluster, and follow instructions.

    ```bash
    scope_config -h
    ```

    This will create a environment-class object that will contain all the relevant information for the job submission. 

1) To **create a system** and add sources (molecules, or unit cells).  

2) To **create a scope input file**, as a plain text file, for easier interaction. 

3) Then, you can run SCOPE in a computation cluster (i.e. a cluster with a job manager, SLURM or SGE), with one -or many- scope input files, and a given system. To do so, you can use the "scope_run_job" script in the computation cluster: 

    ```bash
    scope_run_job -h
    ```

In [ ]:
import os
import scope

In [ ]:
## Path of the data folder. It should be "os.path.abspath('.')+'/Data/Tutorial_4/"
tutorial_folder = os.path.abspath('.')+'/Data/Tutorial_6/'

# Example:

## 1.1) Create Systems: Water, Benzene, Formaldehyde

In [ ]:
## In this example, we create three systems, each with 4 .xyz sources.
from scope.classes_system import System
water_sys   = System("water")
benz_sys    = System("benzene")
form_sys    = System("formaldehyde")

## We set the paths. See Tutorial_4 for more options and information
list_of_systems = []
for tsys in [water_sys, benz_sys, form_sys]:
    tsys.sources_path = f"{tutorial_folder}Sources/{tsys.name}/"
    tsys.calcs_path   = f"{tutorial_folder}Calcs/{tsys.name}/"
    tsys.sys_path     = f"{tutorial_folder}Systems/{tsys.name}/"
    tsys.sys_file     = f"{tutorial_folder}Systems/{tsys.name}/{tsys.name}.npy"
    list_of_systems.append(tsys)

In [ ]:
## We stored 3 systems in a list, with very little information, only the paths. For instance:
print(list_of_systems[0])

## 1.2) Import Molecules in Source Folder 

In [ ]:
# Currently, SCOPE is designed so each system has its own folder with sources. In this example, the sources are .xyz files
# Unfortunately, there is no universal way of creating systems. Each problem in computational chemistry faces its own system structure. 
# In the SCO add-on, the original data is in a cell2mol CELL format, so a new function had to be created to import molecules from there. 
# New add-ons are being prepared, each one with its own form of importing systems, and arranging molecules within those systems.

# For the moment, the simplest way to import sources is when those are in an XYZ format.
for tsys in list_of_systems: 
    tsys.load_multiple_xyz(tsys.sources_path)
    tsys.save()

## If you try to re-run this cell, you'll receive warnings that you're trying to add new sources with existing names. 
## This would be very dangerous. 

In [ ]:
## We now added the molecules as sources. For instance:
print(list_of_systems[0])

In [ ]:
for sou in tsys.sources:
    #print(sou.name, sou.sys_name)
    print(sou)

## 2) Create a Job File - Single point with PBE

In [ ]:
from scope.classes_input import *

In [ ]:
## Job Specs can be created from a file, or from a string. 
## This is an example input read as a string. We will soon go through the different parts
## First, notice that there are four sections, initiated with '&', and terminated with '/'.
## Sections can be empty as in the example below (see &options) or even absent. In those cases, default values will be added
## Also, notice that the values are either introduced as strings 'X', lists [], booleans, or integers/floats...
## Please respect the notation used for each type of variable

test_input = """
&environment
   max_jobs         = 12
   max_procs        = 20
   requested_procs  = 1
/

&options
/

&job_data
   branch       = 'Isolated'
   workflow     = 'all'
   job_setup    = 'regular'
   suffix       = 'scf'
   keyword      = 'pbe scf'

   istate       = 'initial'
   fstate       = 'initial'

   hierarchy    = 1
   requisites   = []
   constrains   = ['self']
   must_be_good = False
/

&qc_data
   software     = 'g16'
   jobtype      = 'scf'
   functional   = 'pbe'
   basis        = 'sto-3g'
   is_grimme    = True
/"""

In [ ]:
# IMPORTANT. Notice that, when running the 'scope_run_job' script. The job file (-j arg) must be an actual file, not a string as in the cell above
# Thus, we save this text to an actual file, so we can call it later
from scope.read_write import save_text
file_name = ''.join([tutorial_folder,"test_job.job"])
save_text(test_input, file_name)

## 3) Run

In [ ]:
## Once the system and a job_file are prepared, we can submit the computation in the appropriate cluster
## To do so:

## 1) Configure the Cluster Environment using the "scope_config" script, installed by default: 

        ## The computation cluster should have Gaussian 16 installed, and a **configured environment**. 
        ## You can give it any name, but for the sake of simplicity, here we will call it "scope_env_cluster.npy" 

## 2) Upload the system file and the test_job.job to the cluster

        ## In principle, you should upload the system files to dedicated folders inside the env.sys_path of the cluster enviromnent (env). 
        ## So, if env.sys_path = '/home/user/scope/tutorials/data/Tutorial_5/Systems', the system file should go to: 
        
        # '/home/user/scope/tutorials/data/Tutorial_5/Systems/water/water.npy'
        # '/home/user/scope/tutorials/data/Tutorial_5/Systems/benzene/benzene.npy'
        # '/home/user/scope/tutorials/data/Tutorial_5/Systems/formaldehyde/formaldehyde.npy'
        
        ## In practice, you can upload the system files anywhere in the cluster. 
        ## However, the cluster's Environment will force the systems to be saved in their expected paths above: 
        ## This is the price to pay for the Environment controlling the paths

        ## The job_file can be saved anywhere. 

## 3) Then, you can use the 'scope_run_job' script.

        ## If the paths are the expected ones, you should type: 

```bash
scope_run_job -n scope_env_cluster.npy -s Systems/water/water.npy -j test_job.job
scope_run_job -n scope_env_cluster.npy -s Systems/benzene/benzene.npy -j test_job.job
scope_run_job -n scope_env_cluster.npy -s Systems/formaldehyde/formaldehyde.npy -j test_job.job
```

## 4) Register

In [1]:
## When executing "scope_run_job", the script takes the indicated (in the job_file) Branch of the system, goes through the computational workflow...
## ... and takes the appropriate actions: 
 
## 1) If the job has not been submitted, it creates the input and submission file and does the submission.
## 2) If the job is running, it does nothing
## 3) If the job has finished, it reads the output and decides whether it finished 'well'
##   3.1) If not, it takes appropriate action. For instance, 
##      3.1.1) If it is a GEOMETRY OPTIMIZATION that didn't reach a zero-gradient solution, it creates and submits a new computation that continues the previous run
##      3.1.2) If it is an SCF computation in Quantum Espresso that didn't converge, it will modify some parameters and resubmit.
##      3.1.3) If the computation was aborted by SLURM or for some other reason, the logfile will be discarded, and the computation will be resubmitted.
##   3.2) If the job finished well, the logfile is parsed, and the relevant data is stored in the final state (fstate) defined by the computation
##        The job will be considered done, and even if you resubmit the same command, the computations won't run again. 

In [3]:
## In tutorial number 6, we will see how those computations are "REGISTERED"